In [1]:
from itertools import combinations
import numpy as np

## Simple Test Problem

In [2]:
CITIES = [
    "Rome",
    "Milan",
    "Naples",
    "Turin",
    "Palermo",
    "Genoa",
    "Bologna",
    "Florence",
    "Bari",
    "Catania",
    "Venice",
    "Verona",
    "Messina",
    "Padua",
    "Trieste",
    "Taranto",
    "Brescia",
    "Prato",
    "Parma",
    "Modena",
]
test_problem = np.load('LAB2-data/test_problem.npy')
test_problem.shape

(20, 20)

## Common tests

In [3]:
problem = np.load('LAB2-data/problem_r2_100.npy')

In [4]:
# Negative values?
np.any(problem < 0)

np.True_

In [5]:
# Diagonal is all zero?
np.allclose(np.diag(problem), 0.0)

False

In [6]:
# Symmetric matrix?
np.allclose(problem, problem.T)

False

### Triangular inequality 
Perché è importante per il lab
Questo test nel notebook tsp.ipynb serve a farti capire la differenza tra i tipi di problema:

I problemi geometrici (g_) rispetteranno questa regola (il risultato sarà True).

I problemi casuali (r1_, r2_) molto probabilmente non la rispetteranno (il risultato sarà False, come puoi vedere dall'output nel notebook per problem_r2_100.npy).

In [7]:
# Triangular inequality
all(
    problem[x, y] <= problem[x, z] + problem[z, y]
    for x, y, z in list(combinations(range(problem.shape[0]), 3))
)

False

In [8]:
import numpy as np
import glob
from itertools import combinations

print("--- TEST DI SIMMETRIA (O(N^2) - Veloce) ---")
print("Controlla se dist(A, B) == dist(B, A)")

# Prendo tutti i file
problem_files = glob.glob("LAB2-data/*.npy")
problem_files = [f for f in problem_files if 'test_problem' not in f]
problem_files.sort()

def get_name(path):
    return path.replace('\\', '/').split('/')[-1]

for problem_file_path in problem_files:
    problem_name = get_name(problem_file_path)
    problem = np.load(problem_file_path)
    
    # np.allclose è il modo corretto per confrontare float
    is_symmetric = np.allclose(problem, problem.T)
    
    print(f"File: {problem_name:<20} | È Simmetrico? -> {is_symmetric}")

print("\nTest di simmetria completato.")

--- TEST DI SIMMETRIA (O(N^2) - Veloce) ---
Controlla se dist(A, B) == dist(B, A)
File: problem_g_10.npy     | È Simmetrico? -> True
File: problem_g_100.npy    | È Simmetrico? -> True
File: problem_g_1000.npy   | È Simmetrico? -> True
File: problem_g_20.npy     | È Simmetrico? -> True
File: problem_g_200.npy    | È Simmetrico? -> True
File: problem_g_50.npy     | È Simmetrico? -> True
File: problem_g_500.npy    | È Simmetrico? -> True
File: problem_r1_10.npy    | È Simmetrico? -> False
File: problem_r1_100.npy   | È Simmetrico? -> False
File: problem_r1_1000.npy  | È Simmetrico? -> False
File: problem_r1_20.npy    | È Simmetrico? -> False
File: problem_r1_200.npy   | È Simmetrico? -> False
File: problem_r1_50.npy    | È Simmetrico? -> False
File: problem_r1_500.npy   | È Simmetrico? -> False
File: problem_r2_10.npy    | È Simmetrico? -> False
File: problem_r2_100.npy   | È Simmetrico? -> False
File: problem_r2_1000.npy  | È Simmetrico? -> False
File: problem_r2_20.npy    | È Simmetrico

## HILL CLIMBING

**Obbiettivo:** Trovare il percorso con il "punteggio" (costo) più **basso** possibile.

- **_100** (il numero): Indica la dimensione del problema, cioè il numero di città. In questo caso, 100 città. Troverai file con 10, 20, 50, 100, 200, 500, e 1000 città.

- **g_** (la lettera): Indica la natura della matrice delle distanze.

- **g (Geometric)**: Il problema è geometrico e simmetrico. Significa che la distanza da A a B è uguale alla distanza da B ad A. Queste matrici rispettano anche la disuguaglianza triangolare (andare da A a C non è mai più lungo che passare da A a B e poi a C).

- **r1 e r2 (Random)**: I problemi sono asimmetrici. Significa che la distanza da A a B è diversa dalla distanza da B ad A (pensa a un volo aereo o a strade a senso unico). Come puoi vedere nel notebook tsp.ipynb di partenza, il test np.allclose(problem, problem.T) fallisce per problem_r2_100.npy, confermando che la matrice non è simmetrica. r1 e r2 sono semplicemente due generatori diversi di problemi asimmetrici.



In breve:

g: TSP Simmetrico (facile)

r1, r2: TSP Asimmetrico (difficile)

Numero: Quante città.

In [9]:
import numpy as np
import random
import time
import glob
import pandas as pd
import math
from IPython.display import display




# --- Funzioni di Supporto---

def evaluate_solution(solution, distance_matrix):
    """
    Calcola il costo totale (distanza) di un percorso (soluzione).
    Operazione O(N).
    """
    total_cost = 0
    num_cities = len(solution)
    for i in range(num_cities):
        city_a = solution[i]
        city_b = solution[(i + 1) % num_cities] # % gestisce il ritorno all'inizio
        total_cost += distance_matrix[city_a, city_b]
    return total_cost

def get_random_solution(num_cities):
    """
    Genera una soluzione casuale (un percorso come lista di indici).
    """
    solution = list(range(num_cities))
    random.shuffle(solution)
    return solution



In [10]:
# --- Algoritmo 1: SA CLASSICO (Corretto per Asimmetrici, O(N) per mossa) ---

def simulated_annealing_2opt_CLASSIC(distance_matrix):
    """
    Esegue il SA ricalcolando l'intero costo ad ogni mossa.
    Corretto per tutti i problemi (g_, r1_, r2_).
    """
    num_cities = distance_matrix.shape[0]
    
    T_initial = 1000.0
    T_min = 0.1
    cooling_rate = 0.999
    iterations_per_temp = num_cities * 2 
    
    current_solution = get_random_solution(num_cities)
    current_cost = evaluate_solution(current_solution, distance_matrix)
    
    best_solution = current_solution
    best_cost = current_cost
    
    T = T_initial
    
    while T > T_min:
        for _ in range(iterations_per_temp):
            
            i, k = random.sample(range(num_cities), 2)
            if i > k:
                i, k = k, i 
            
            neighbor_solution = current_solution[:i] + current_solution[i:k+1][::-1] + current_solution[k+1:]
            neighbor_cost = evaluate_solution(neighbor_solution, distance_matrix)
            
            delta_cost = neighbor_cost - current_cost
            
            if delta_cost < 0 or random.random() < math.exp(-delta_cost / T):
                current_solution = neighbor_solution
                current_cost = neighbor_cost
                
                if current_cost < best_cost:
                    best_solution = current_solution
                    best_cost = current_cost
        
        T *= cooling_rate
            
    return best_solution, best_cost


'''
# --- Algoritmo 2: SA FAST (Ottimizzato per Simmetrici, O(1) per mossa) ---

def simulated_annealing_2opt_FAST(distance_matrix):
    """
    Esegue il SA usando una valutazione delta O(1).
    Veloce ma CORRETTO SOLO per problemi SIMMETRICi (g_).
    """
    num_cities = distance_matrix.shape[0]
    
    T_initial = 100.0
    T_min = 0.01
    cooling_rate = 0.9999
    iterations_per_temp = num_cities * 10
    
    total_iterations = 0
    MAX_ITERATIONS = 2_000_000 
    
    current_solution = get_random_solution(num_cities)
    current_cost = evaluate_solution(current_solution, distance_matrix)
    
    best_solution = current_solution
    best_cost = current_cost
    
    T = T_initial

    while T > T_min and total_iterations < MAX_ITERATIONS:
        for _ in range(iterations_per_temp):
            total_iterations += 1
            
            i, k = random.sample(range(num_cities), 2)
            if i > k:
                i, k = k, i 
            
            city_i_minus_1 = current_solution[i - 1]
            city_i = current_solution[i]
            city_k = current_solution[k]
            city_k_plus_1 = current_solution[(k + 1) % num_cities]
            
            cost_removed = distance_matrix[city_i_minus_1, city_i] + distance_matrix[city_k, city_k_plus_1]
            cost_added = distance_matrix[city_i_minus_1, city_k] + distance_matrix[city_i, city_k_plus_1]
            
            delta_cost = cost_added - cost_removed
            
            if delta_cost < 0 or random.random() < math.exp(-delta_cost / T):
                current_solution = current_solution[:i] + current_solution[i:k+1][::-1] + current_solution[k+1:]
                current_cost += delta_cost
                
                if current_cost < best_cost:
                    best_solution = current_solution
                    best_cost = current_cost
        
        T *= cooling_rate
            
    return best_solution, best_cost
'''

####

# --- Algoritmo 2: SA FAST (Ottimizzato per Simmetrici, O(1) per mossa) ---

def simulated_annealing_2opt_FAST(distance_matrix):
    """
    Esegue il SA usando una valutazione delta O(1).
    Veloce e CORRETTO SOLO per problemi SIMMETRICI (g_).
    """
    num_cities = distance_matrix.shape[0]
    
    T_initial = 100.0
    T_min = 0.01
    cooling_rate = 0.9999
    iterations_per_temp = num_cities * 10
    
    total_iterations = 0
    MAX_ITERATIONS = 2_000_000 
    
    current_solution = get_random_solution(num_cities)
    current_cost = evaluate_solution(current_solution, distance_matrix)
    
    best_solution = current_solution
    best_cost = current_cost
    
    T = T_initial

    while T > T_min and total_iterations < MAX_ITERATIONS:
        for _ in range(iterations_per_temp):
            total_iterations += 1
            
            i, k = random.sample(range(num_cities), 2)
            if i > k:
                i, k = k, i
            
            # --- BUG CORRETTO ---
            # Evita il caso i=0, k=N-1 che rompe lo stesso arco (wrap-around)
            if i == 0 and k == num_cities - 1:
                continue
            # --------------------
            
            city_i_minus_1 = current_solution[i - 1]
            city_i = current_solution[i]
            city_k = current_solution[k]
            city_k_plus_1 = current_solution[(k + 1) % num_cities]
            
            cost_removed = distance_matrix[city_i_minus_1, city_i] + distance_matrix[city_k, city_k_plus_1]
            cost_added = distance_matrix[city_i_minus_1, city_k] + distance_matrix[city_i, city_k_plus_1]
            
            delta_cost = cost_added - cost_removed
            
            if delta_cost < 0 or random.random() < math.exp(-delta_cost / T):
                current_solution = current_solution[:i] + current_solution[i:k+1][::-1] + current_solution[k+1:]
                current_cost += delta_cost
                
                if current_cost < best_cost:
                    best_solution = current_solution
                    best_cost = current_cost
        
        T *= cooling_rate
            
    return best_solution, best_cost

### Divisione dei problemi 

In [ ]:
# Cerca tutti i file e li divide in categorie
problem_files = glob.glob("LAB2-data/*.npy")
problem_files = [f for f in problem_files if 'test_problem' not in f]

# Funzione helper per estrarre il nome pulito
def get_name(path):
    return path.replace('\\', '/').split('/')[-1]

# Divide i file in 3 liste
g_files = sorted([f for f in problem_files if get_name(f).startswith('problem_g')])
r1_files = sorted([f for f in problem_files if get_name(f).startswith('problem_r1')])
r2_files = sorted([f for f in problem_files if get_name(f).startswith('problem_r2')])

print(f"File G (Simmetrici): {len(g_files)}")
print(f"File R1 (Asimmetrici): {len(r1_files)}")
print(f"File R2 (Asimmetrici): {len(r2_files)}")



ordina bene crescente per numero di città

File G (Simmetrici): 7
File R1 (Asimmetrici): 7
File R2 (Asimmetrici): 7


In [12]:
print("\n--- ESECUZIONE PROBLEMI GEOMETRICI (g_) ---")
g_results = []

for problem_file_path in g_files:
    problem_name = get_name(problem_file_path)
    problem_matrix = np.load(problem_file_path)
    
    print(f"\n Processando (SA-FAST - O(1)): {problem_name}")
    start_time = time.time()
    sa_sol, sa_cost = simulated_annealing_2opt_FAST(problem_matrix)
    sa_time = time.time() - start_time
    
    print(f"    -> Risultato: Costo={sa_cost:.2f}, Tempo={sa_time:.4f}s")
    
    g_results.append({
        'Problem': problem_name,
        'SA Cost': sa_cost,
        'SA Time (s)': sa_time
    })

# Stampa la tabella riassuntiva per questa categoria
print("\n--- Risultati Categoria G ---")
g_df = pd.DataFrame(g_results)
display(g_df)


--- ESECUZIONE PROBLEMI GEOMETRICI (g_) ---

 Processando (SA-FAST - O(1)): problem_g_10.npy
    -> Risultato: Costo=1497.66, Tempo=2.6147s

 Processando (SA-FAST - O(1)): problem_g_100.npy
    -> Risultato: Costo=10030.19, Tempo=2.8310s

 Processando (SA-FAST - O(1)): problem_g_1000.npy
    -> Risultato: Costo=131411.78, Tempo=7.0742s

 Processando (SA-FAST - O(1)): problem_g_20.npy
    -> Risultato: Costo=1755.51, Tempo=2.6419s

 Processando (SA-FAST - O(1)): problem_g_200.npy
    -> Risultato: Costo=22204.12, Tempo=3.3751s

 Processando (SA-FAST - O(1)): problem_g_50.npy
    -> Risultato: Costo=4172.52, Tempo=2.7525s

 Processando (SA-FAST - O(1)): problem_g_500.npy
    -> Risultato: Costo=62568.99, Tempo=5.9818s

--- Risultati Categoria G ---


,Problem,SA Cost,SA Time (s)
0,problem_g_10.npy,1497.663648,2.614726
1,problem_g_100.npy,10030.187421,2.830988
2,problem_g_1000.npy,131411.777785,7.074162
3,problem_g_20.npy,1755.514677,2.641852
4,problem_g_200.npy,22204.122748,3.375067
5,problem_g_50.npy,4172.515932,2.752487
6,problem_g_500.npy,62568.993025,5.981782


In [13]:
print("\n--- ESECUZIONE PROBLEMI ASIMMETRICI (r1_) ---")
r1_results = []

for problem_file_path in r1_files:
    problem_name = get_name(problem_file_path)
    problem_matrix = np.load(problem_file_path)
    
    print(f"\n Processando (SA-CLASSIC - O(N)): {problem_name}")
    start_time = time.time()
    sa_sol, sa_cost = simulated_annealing_2opt_CLASSIC(problem_matrix)
    sa_time = time.time() - start_time
    
    print(f"    -> Risultato: Costo={sa_cost:.2f}, Tempo={sa_time:.4f}s")
    
    r1_results.append({
        'Problem': problem_name,
        'SA Cost': sa_cost,
        'SA Time (s)': sa_time
    })

# Stampa la tabella riassuntiva per questa categoria
print("\n--- Risultati Categoria R1 ---")
r1_df = pd.DataFrame(r1_results)
display(r1_df)


--- ESECUZIONE PROBLEMI ASIMMETRICI (r1_) ---

 Processando (SA-CLASSIC - O(N)): problem_r1_10.npy
    -> Risultato: Costo=184.27, Tempo=0.3910s

 Processando (SA-CLASSIC - O(N)): problem_r1_100.npy
    -> Risultato: Costo=1159.76, Tempo=18.7304s

 Processando (SA-CLASSIC - O(N)): problem_r1_1000.npy
    -> Risultato: Costo=12923.40, Tempo=3132.0161s

 Processando (SA-CLASSIC - O(N)): problem_r1_20.npy
    -> Risultato: Costo=359.08, Tempo=1.1291s

 Processando (SA-CLASSIC - O(N)): problem_r1_200.npy
    -> Risultato: Costo=2353.69, Tempo=70.8517s

 Processando (SA-CLASSIC - O(N)): problem_r1_50.npy
    -> Risultato: Costo=749.02, Tempo=5.1737s

 Processando (SA-CLASSIC - O(N)): problem_r1_500.npy
    -> Risultato: Costo=5724.41, Tempo=990.8948s

--- Risultati Categoria R1 ---


,Problem,SA Cost,SA Time (s)
0,problem_r1_10.npy,184.273441,0.391011
1,problem_r1_100.npy,1159.762522,18.730389
2,problem_r1_1000.npy,12923.402849,3132.016091
3,problem_r1_20.npy,359.080861,1.129131
4,problem_r1_200.npy,2353.693787,70.851748
5,problem_r1_50.npy,749.024386,5.173698
6,problem_r1_500.npy,5724.412664,990.894758


In [14]:
print("\n--- ESECUZIONE PROBLEMI ASIMMETRICI (r2_) ---")
r2_results = []

for problem_file_path in r2_files:
    problem_name = get_name(problem_file_path)
    problem_matrix = np.load(problem_file_path)
    
    print(f"\n Processando (SA-CLASSIC - O(N)): {problem_name}")
    start_time = time.time()
    sa_sol, sa_cost = simulated_annealing_2opt_CLASSIC(problem_matrix)
    sa_time = time.time() - start_time
    
    print(f"    -> Risultato: Costo={sa_cost:.2f}, Tempo={sa_time:.4f}s")
    
    r2_results.append({
        'Problem': problem_name,
        'SA Cost': sa_cost,
        'SA Time (s)': sa_time
    })

# Stampa la tabella riassuntiva per questa categoria
print("\n--- Risultati Categoria R2 ---")
r2_df = pd.DataFrame(r2_results)
display(r2_df)


--- ESECUZIONE PROBLEMI ASIMMETRICI (r2_) ---

 Processando (SA-CLASSIC - O(N)): problem_r2_10.npy
    -> Risultato: Costo=-402.92, Tempo=0.3879s

 Processando (SA-CLASSIC - O(N)): problem_r2_100.npy
    -> Risultato: Costo=-3481.04, Tempo=18.2624s

 Processando (SA-CLASSIC - O(N)): problem_r2_1000.npy


KeyboardInterrupt: 

In [ ]:
print("\n\n--- ANALISI FINALE (Tabella Riassuntiva) ---")

# Lista per contenere i DataFrame che esistono
all_dfs = []

# 'locals()' controlla quali variabili sono state definite
if 'g_df' in locals():
    all_dfs.append(g_df)
if 'r1_df' in locals():
    all_dfs.append(r1_df)
if 'r2_df' in locals():
    all_dfs.append(r2_df)

if all_dfs:
    # Concatena tutti i DataFrame esistenti
    final_df = pd.concat(all_dfs, ignore_index=True)
    display(final_df)
else:
    print("Esegui le celle 3, 4, e/o 5 per generare i risultati.")

In [ ]:

''' 

vedi prima lezione lab ( dura poco)
stmpali per vedere ordine
fai solo SA ottimixzato ( no greedy)  ---> i dataset più grandi -- molto difficile avere una buona soluzione
- triangle inequality -- ??? a che serve , vedi bene 


START WITH:

1. "test problem" per capire un po come funziona

'''